# 학습 곡선과 과적합 테스트 (pegstar)

1. 한 실험의 **train / val 손실 선그래프**, 검증 mAP, 실제 학습률
2. 과적합 지표: val 손실이 가장 낮았던 epoch 이후 다시 올랐는지, train/val 차이
3. 여러 실험(모델) 곡선 비교
4. **K-fold · seed** 결과: 점수 분포(평균 ± 표준편차)와 fold별 곡선

- 데이터: 학습마다 YOLO가 남기는 `outputs/{실험}/yolo_run/results.csv` (epoch별 기록)
- K-fold·seed 실행: `scripts/kfold_split.py` → `scripts/run_kfold.py` (요약: `outputs/kfold/*_summary.csv`)
- 곡선의 mAP는 YOLO 자체 지표(mAP50, mAP50-95)다. 대회 지표 mAP@[0.75:0.95]는 학습 끝의 `metrics.json`·요약 CSV로 본다
- 그림은 `reports/figures/`에 저장 (숫자 그래프라 PR에 넣어도 됨)

In [ ]:
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / "outputs"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

CLOSE_MOSAIC = 10  # Ultralytics 기본값: 마지막 10 epoch는 모자이크 증강을 끈다 → train 손실이 갑자기 떨어질 수 있음
LOSSES = ["box", "cls", "dfl"]
TRAIN_C, VAL_C = "#2a6fdb", "#e8590c"


def load_curve(exp: str) -> pd.DataFrame:
    df = pd.read_csv(OUT / exp / "yolo_run" / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    return df


def overfit_report(exp: str) -> pd.DataFrame:
    """손실별: val 최저 epoch, 그 뒤로 얼마나 다시 올랐는지, 마지막 train/val 차이."""
    df = load_curve(exp)
    last = df.iloc[-1]
    rows = []
    for k in LOSSES:
        val = df[f"val/{k}_loss"]
        best = int(df.loc[val.idxmin(), "epoch"])
        rise = (val.iloc[-1] - val.min()) / val.min() * 100
        rows.append(
            {
                "실험": exp,
                "손실": k,
                "val 최저 epoch": best,
                "마지막 epoch": int(last["epoch"]),
                "val 최저": round(val.min(), 4),
                "val 마지막": round(val.iloc[-1], 4),
                "최저 대비 상승(%)": round(rise, 1),
                "마지막 val-train": round(last[f"val/{k}_loss"] - last[f"train/{k}_loss"], 4),
                "판단": "과적합 신호" if rise > 3 and best < last["epoch"] - 2 else "-",
            }
        )
    return pd.DataFrame(rows)

## 1. 한 실험의 학습 곡선
주황 = val, 파랑 = train. 회색 영역은 모자이크 증강이 꺼지는 마지막 구간, 점은 val 손실이 가장 낮았던 epoch.

In [ ]:
EXP = "1027_pegstar_yolov8m_v0_mcompare"  # 보고 싶은 실험 이름으로 바꾸기

df = load_curve(EXP)
ep = df["epoch"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
for ax, k in zip([axes[0, 0], axes[0, 1], axes[1, 0]], LOSSES):
    ax.plot(ep, df[f"train/{k}_loss"], color=TRAIN_C, label="train")
    ax.plot(ep, df[f"val/{k}_loss"], color=VAL_C, label="val")
    i = df[f"val/{k}_loss"].idxmin()
    ax.scatter(ep[i], df.loc[i, f"val/{k}_loss"], color=VAL_C, zorder=3)
    ax.set_title(f"{k} 손실 (로그 눈금)")
    ax.set_yscale("log")  # 초반 큰 값 때문에 후반 차이가 안 보이지 않도록
for ax in axes.flat[:3]:
    ax.axvspan(ep.max() - CLOSE_MOSAIC + 0.5, ep.max(), color="0.9", zorder=0)
    ax.set_xlabel("epoch")
    ax.legend(loc="upper right")

ax = axes[1, 1]
ax.plot(ep, df["metrics/mAP50(B)"], label="mAP50", color="#9db8e8")
ax.plot(ep, df["metrics/mAP50-95(B)"], label="mAP50-95", color=TRAIN_C)
ax.set_ylim(0, 1.02)
ax.set_xlabel("epoch")
ax.set_title("검증 mAP (YOLO 지표)")
ax.legend(loc="center right")
ax2 = ax.twinx()
ax2.plot(ep, df["lr/pg0"], color="0.5", ls="--", label="학습률")
ax2.set_ylabel("실제 학습률 (lr/pg0)", color="0.4")
ax2.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
for a in axes.flat:
    a.spines[["top"]].set_visible(False)
fig.suptitle(f"{EXP} 학습 곡선  (실제 최대 학습률 {df['lr/pg0'].max():.2e})", fontsize=12)
plt.tight_layout()
plt.savefig(FIG / f"curves_{EXP}.png", dpi=150, bbox_inches="tight")

overfit_report(EXP)

**읽는 법**
- train은 계속 내려가는데 val이 최저점 이후 다시 오르면 → 과적합 신호 (표의 `최저 대비 상승(%)`이 3% 넘으면 표시)
- 마지막 10 epoch(회색)에서 train이 갑자기 떨어지는 건 모자이크가 꺼져서일 수 있다. 이 구간만의 변화는 과적합으로 단정하지 않는다
- 학습률 그래프가 config의 `lr`보다 훨씬 작다면 `optimizer: auto`가 학습률을 스스로 정한 것이다 (config의 `lr`은 무시됨)

## 2. 여러 실험 곡선 비교

In [ ]:
EXPS = {
    "YOLOv8m": "1027_pegstar_yolov8m_v0_mcompare",
    "YOLO11m": "1027_pegstar_yolo11m_v0_mcompare",
    "YOLO12m": "1027_pegstar_yolo12m_v0_mcompare",
    "YOLO26m": "1027_pegstar_yolo26m_v0_mcompare",
}
EXPS = {k: v for k, v in EXPS.items() if (OUT / v / "yolo_run" / "results.csv").exists()}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, exp in EXPS.items():
    d = load_curve(exp)
    axes[0].plot(d["epoch"], d["val/box_loss"], label=name)
    axes[1].plot(d["epoch"], d["val/cls_loss"], label=name)
    axes[1].set_yscale("log")
    axes[2].plot(d["epoch"], d["metrics/mAP50-95(B)"], label=name)
for ax, t in zip(axes, ["val box 손실", "val cls 손실", "검증 mAP50-95"]):
    ax.set_title(t)
    ax.set_xlabel("epoch")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend()
plt.tight_layout()
plt.savefig(FIG / "curves_compare.png", dpi=150, bbox_inches="tight")

pd.concat([overfit_report(e) for e in EXPS.values()]).query("손실 == 'cls'")

## 3. K-fold · seed 결과
`scripts/run_kfold.py`를 돌린 뒤 실행. 요약 CSV마다 실행별 점수(점)와 평균 ± 표준편차(막대)를 그린다.

In [ ]:
summaries = sorted((OUT / "kfold").glob("*_summary.csv"))
if not summaries:
    print("아직 K-fold/seed 결과가 없습니다. 먼저 실행:")
    print("  uv run python scripts/kfold_split.py --k 5")
    print(
        "  uv run python scripts/run_kfold.py --config configs/yolo_m_v8m.yaml --folds 0 1 2 3 4 --tag kfold"
    )
    print(
        "  uv run python scripts/run_kfold.py --config configs/yolo_m_v8m.yaml --seeds 42 0 1 --tag seed"
    )
else:
    tables = {p.stem.replace("_summary", ""): pd.read_csv(p) for p in summaries}
    stat = pd.DataFrame(
        [
            {
                "요약": k,
                "실행 수": len(t),
                "mAP@[0.75:0.95] 평균": t["map"].mean(),
                "표준편차": t["map"].std(),
                "최소": t["map"].min(),
                "최대": t["map"].max(),
            }
            for k, t in tables.items()
        ]
    ).round(4)

    fig, ax = plt.subplots(figsize=(1.8 + 1.6 * len(tables), 4))
    for i, t in enumerate(tables.values()):
        ax.scatter([i] * len(t), t["map"], color=TRAIN_C, alpha=0.7, zorder=3)
        ax.errorbar(
            i + 0.15,
            t["map"].mean(),
            yerr=t["map"].std() if len(t) > 1 else 0,
            fmt="o",
            color=VAL_C,
            capsize=6,
            zorder=3,
        )
        ax.text(
            i + 0.22,
            t["map"].mean(),
            f"{t['map'].mean():.3f}±{t['map'].std() if len(t) > 1 else 0:.3f}",
            va="center",
            fontsize=9,
        )
    ax.set_xticks(range(len(tables)), [k.split("_")[-1] for k in tables])
    ax.set_xlim(-0.5, len(tables) - 0.2)
    ax.set_ylabel("mAP@[0.75:0.95]")
    ax.set_title("실행별 점수(파랑)와 평균 ± 표준편차(주황)")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIG / "kfold_scores.png", dpi=150, bbox_inches="tight")
    display(stat)

In [ ]:
# fold(또는 seed)별 학습 곡선 겹쳐 보기 + 과적합 표
if summaries:
    KEY = list(tables)[0]  # 다른 요약을 보려면 이름으로 바꾸기
    runs = [e for e in tables[KEY]["exp_name"] if (OUT / e / "yolo_run" / "results.csv").exists()]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for e in runs:
        d = load_curve(e)
        label = e.split("_")[-2] + "_" + e.split("_")[-1]
        axes[0].plot(d["epoch"], d["val/cls_loss"], label=label)
        axes[0].plot(d["epoch"], d["train/cls_loss"], color="0.75", lw=0.8)
        axes[1].plot(d["epoch"], d["metrics/mAP50-95(B)"], label=label)
    axes[0].set_title("cls 손실: val(색) / train(회색)")
    axes[1].set_title("검증 mAP50-95")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.legend(fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(KEY)
    plt.tight_layout()
    plt.savefig(FIG / f"kfold_curves_{KEY}.png", dpi=150, bbox_inches="tight")
    display(pd.concat([overfit_report(e) for e in runs]))

In [ ]:
EXP = "1027_pegstar_yolov8m_v0_mcompare_kfold_v0_f0_s42"
df = load_curve(EXP)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: train vs val 손실 (세 손실을 더한 값)
train_total = df["train/box_loss"] + df["train/cls_loss"] + df["train/dfl_loss"]
val_total = df["val/box_loss"] + df["val/cls_loss"] + df["val/dfl_loss"]
axes[0].plot(df["epoch"], train_total, label="train", color=TRAIN_C)
axes[0].plot(df["epoch"], val_total, label="val", color=VAL_C)
axes[0].set_title("train vs val 전체 손실")
axes[0].set_xlabel("epoch")
axes[0].legend()

# 오른쪽: 실제 학습률 (train에만 있음)
axes[1].plot(df["epoch"], df["lr/pg0"], color="0.4")
axes[1].set_title("학습률 (epoch별)")
axes[1].set_xlabel("epoch")
axes[1].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG / f"loss_lr_{EXP}.png", dpi=150, bbox_inches="tight")

## 결론 쓰는 법
- **K-fold 표준편차**가 모델 간 점수 차이보다 크면 → 그 차이는 분할 운일 가능성이 크다
- **seed 표준편차**는 같은 조건에서도 생기는 흔들림 → 실험 비교 때 "의미 있는 차이"의 최소 기준
- fold 대부분에서 val 손실이 최저점 이후 오르면 → epoch를 줄이거나(early stopping, `yolo_args: {patience: ...}`) 증강·정규화 강화